# MISMIP mediator experiment suite visualisation

Catalog-driven comparison notebook for the NUOPC-driven MISMIP melt-rate and coupling-cadence experiments. Run `run_mediator_experiment_suite.py` first so the catalog and per-experiment `result.nc` files exist.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import netCDF4 as nc
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

CATALOG_PATH = Path('/scratch/au88/jr5971/issm_simple_mediator/experiments/catalog.csv')
CONTROL_ID = 'melt_0_cpl10y'
FIELDS = ['Thickness', 'Surface', 'Vel', 'MaskOceanLevelset']
assert CATALOG_PATH.exists(), CATALOG_PATH

## Catalog

In [ ]:
catalog = pd.read_csv(CATALOG_PATH)
for column in ['melt_rate_m_per_s', 'melt_rate_m_per_yr', 'coupling_years', 'final_time_years', 'output_frequency_years', 'run_steps']:
    if column in catalog:
        catalog[column] = pd.to_numeric(catalog[column], errors='coerce')

catalog['result_exists'] = catalog['result_nc'].apply(lambda value: Path(str(value)).exists())
display_columns = [
    'experiment_id', 'suite', 'status', 'purpose', 'melt_rate_m_per_yr',
    'coupling_years', 'final_time_years', 'output_frequency_years',
    'run_steps', 'result_exists', 'mediator_log',
]
display(catalog[[col for col in display_columns if col in catalog.columns]])

## Load Results

In [ ]:
def load_mismip_result(path):
    with nc.Dataset(path) as ds:
        mesh = ds.groups['mesh']
        result = ds.groups['results'].groups['TransientSolution']

        x = np.asarray(mesh.variables['x'][:], dtype=float)
        y = np.asarray(mesh.variables['y'][:], dtype=float)
        elements = np.asarray(mesh.variables['elements'][:], dtype=int) - 1
        triang = mtri.Triangulation(x, y, elements)

        time = np.asarray(result.variables['time'][:], dtype=float)
        step = np.asarray(result.variables['step'][:], dtype=int)
        fields = {}
        for name, var in result.variables.items():
            arr = np.asarray(var[:])
            if arr.ndim == 2 and arr.shape[1] == x.size:
                fields[name] = arr.astype(float)

    return {'x': x, 'y': y, 'elements': elements, 'triang': triang, 'time': time, 'step': step, 'fields': fields}

results = {}
for row in catalog.itertuples(index=False):
    result_path = Path(str(row.result_nc))
    if str(getattr(row, 'status', '')) == 'complete' and result_path.exists():
        results[row.experiment_id] = load_mismip_result(result_path)

print(f'Loaded {len(results)} completed experiments: {list(results)}')

summary_rows = []
for exp_id, data in results.items():
    summary_rows.append({
        'experiment_id': exp_id,
        'saved_output_count': len(data['time']),
        'first_time_yr': data['time'][0] if len(data['time']) else np.nan,
        'final_time_yr': data['time'][-1] if len(data['time']) else np.nan,
        'fields': ', '.join(sorted(data['fields'])),
    })
display(pd.DataFrame(summary_rows))

## Final-State Maps

In [ ]:
def plot_final_maps(experiment_ids=None, fields=FIELDS):
    experiment_ids = experiment_ids or list(results)
    for field in fields:
        available = [exp_id for exp_id in experiment_ids if field in results[exp_id]['fields']]
        if not available:
            print(f'{field}: no completed experiments contain this field')
            continue
        fig, axes = plt.subplots(1, len(available), figsize=(4.2 * len(available), 3.8), constrained_layout=True)
        axes = np.atleast_1d(axes)
        for ax, exp_id in zip(axes, available):
            data = results[exp_id]
            values = data['fields'][field][-1]
            im = ax.tripcolor(data['triang'], values, shading='gouraud')
            ax.set_aspect('equal')
            ax.set_title(f'{exp_id}\n{field}, year {data["time"][-1]:g}')
            ax.set_xlabel('x (m)')
            ax.set_ylabel('y (m)')
            fig.colorbar(im, ax=ax, shrink=0.85)
        plt.show()

plot_final_maps()

## Final Minus Control

In [ ]:
def plot_differences_to_control(control_id=CONTROL_ID, fields=FIELDS):
    if control_id not in results:
        print(f'Control {control_id} is not loaded yet.')
        return
    comparison_ids = [exp_id for exp_id in results if exp_id != control_id]
    for field in fields:
        if field not in results[control_id]['fields']:
            continue
        available = [exp_id for exp_id in comparison_ids if field in results[exp_id]['fields']]
        if not available:
            continue
        fig, axes = plt.subplots(1, len(available), figsize=(4.2 * len(available), 3.8), constrained_layout=True)
        axes = np.atleast_1d(axes)
        control = results[control_id]['fields'][field][-1]
        for ax, exp_id in zip(axes, available):
            data = results[exp_id]
            delta = data['fields'][field][-1] - control
            lim = np.nanmax(np.abs(delta)) or 1.0
            im = ax.tripcolor(data['triang'], delta, shading='gouraud', cmap='RdBu_r', vmin=-lim, vmax=lim)
            ax.set_aspect('equal')
            ax.set_title(f'{exp_id} - {control_id}\n{field}')
            ax.set_xlabel('x (m)')
            ax.set_ylabel('y (m)')
            fig.colorbar(im, ax=ax, shrink=0.85)
        plt.show()

plot_differences_to_control()

## Domain-Mean Time Series

In [ ]:
def plot_domain_mean_timeseries(fields=('Thickness', 'Surface', 'Vel')):
    for field in fields:
        fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
        any_field = False
        for exp_id, data in results.items():
            if field not in data['fields']:
                continue
            any_field = True
            mean_values = np.nanmean(data['fields'][field], axis=1)
            ax.plot(data['time'], mean_values, marker='o', label=exp_id)
        if not any_field:
            plt.close(fig)
            continue
        ax.set_title(f'Domain mean {field}')
        ax.set_xlabel('model time (yr)')
        ax.set_ylabel(field)
        ax.legend(fontsize=8)
        plt.show()

plot_domain_mean_timeseries()

## Centerline Profiles

In [ ]:
def centerline_indices(data, y_fraction=0.5, tolerance=None):
    y = data['y']
    target = y.min() + y_fraction * (y.max() - y.min())
    if tolerance is None:
        tolerance = 0.02 * (y.max() - y.min())
    mask = np.abs(y - target) <= tolerance
    if mask.sum() < 3:
        mask = np.argsort(np.abs(y - target))[:max(3, min(50, y.size))]
        order = np.argsort(data['x'][mask])
        return np.asarray(mask)[order]
    idx = np.where(mask)[0]
    return idx[np.argsort(data['x'][idx])]

def plot_centerlines(field='Thickness', control_id=CONTROL_ID):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    control_profile = None
    control_x = None
    if control_id in results and field in results[control_id]['fields']:
        cidx = centerline_indices(results[control_id])
        control_x = results[control_id]['x'][cidx]
        control_profile = results[control_id]['fields'][field][-1, cidx]

    for exp_id, data in results.items():
        if field not in data['fields']:
            continue
        idx = centerline_indices(data)
        x_km = data['x'][idx] / 1000
        profile = data['fields'][field][-1, idx]
        axes[0].plot(x_km, profile, label=exp_id)
        if control_profile is not None and exp_id != control_id:
            delta = np.interp(data['x'][idx], control_x, control_profile)
            axes[1].plot(x_km, profile - delta, label=exp_id)

    axes[0].set_title(f'Final centerline {field}')
    axes[0].set_xlabel('x (km)')
    axes[0].set_ylabel(field)
    axes[1].set_title(f'Centerline difference from {control_id}')
    axes[1].set_xlabel('x (km)')
    axes[1].set_ylabel(f'delta {field}')
    for ax in axes:
        ax.legend(fontsize=8)
    plt.show()

plot_centerlines('Thickness')
plot_centerlines('Surface')

## Coupling-Cadence Comparison

In [ ]:
def plot_coupling_comparison(field='Thickness'):
    pair = ['melt_1myr_cpl10y', 'melt_1myr_cpl5y']
    if not all(exp_id in results and field in results[exp_id]['fields'] for exp_id in pair):
        print(f'Need completed {pair} results with {field} to compare coupling cadence.')
        return
    a, b = (results[exp_id] for exp_id in pair)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    for ax, exp_id, data in zip(axes, pair, [a, b]):
        ax.plot(data['time'], np.nanmean(data['fields'][field], axis=1), marker='o')
        ax.set_title(exp_id)
        ax.set_xlabel('model time (yr)')
        ax.set_ylabel(f'domain mean {field}')
    plt.show()

    delta = b['fields'][field][-1] - a['fields'][field][-1]
    lim = np.nanmax(np.abs(delta)) or 1.0
    fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
    im = ax.tripcolor(b['triang'], delta, shading='gouraud', cmap='RdBu_r', vmin=-lim, vmax=lim)
    ax.set_aspect('equal')
    ax.set_title(f'{pair[1]} - {pair[0]} final {field}')
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.show()

plot_coupling_comparison('Thickness')
plot_coupling_comparison('Surface')

## Mediator Clock And Provenance

In [ ]:
def parse_mediator_log(path):
    path = Path(str(path))
    if not path.exists():
        return pd.DataFrame()
    rows = []
    clock_re = re.compile(r'(?:timeStepSeconds|time_step_seconds|time step seconds)[^0-9]*(\d+)', re.I)
    step_re = re.compile(r'(?:step|advance)[^0-9]*(\d+)', re.I)
    for line_no, line in enumerate(path.read_text(errors='replace').splitlines(), start=1):
        if any(token in line for token in ['timeStep', 'time_step', 'advance', 'ModelAdvance', 'clock']):
            rows.append({'line': line_no, 'text': line.strip()[:240]})
    return pd.DataFrame(rows)

log_rows = []
for row in catalog.itertuples(index=False):
    log_path = Path(str(getattr(row, 'mediator_log', '')))
    parsed = parse_mediator_log(log_path)
    log_rows.append({
        'experiment_id': row.experiment_id,
        'mediator_log_exists': log_path.exists(),
        'matched_clock_lines': len(parsed),
        'mediator_log': str(log_path),
    })

display(pd.DataFrame(log_rows))
for row in catalog.itertuples(index=False):
    parsed = parse_mediator_log(getattr(row, 'mediator_log', ''))
    if len(parsed):
        print(f'--- {row.experiment_id} ---')
        display(parsed.head(20))